# Scalability — Horizontal vs Vertical Scaling

## 🧠 Mental Model

> **Vertical scaling = buying a bigger truck. Horizontal scaling = hiring more trucks.
> The bigger truck has limits; at some size, the cost doubles but capacity doesn't.
> More trucks can grow forever — but now you need a dispatcher.**

### WHY Scalability Exists as a Design Concern

Single-server systems fail in two ways:
1. **Capacity ceiling** — hardware limits (max RAM a single machine supports ~12TB, max cores ~128)
2. **SPOF** — one machine = one point of failure

Scalability is the property of a system that allows it to **handle growing load** by
adding resources, with minimal rework of the core design.

### The Three Axes of Scale (The Scale Cube)

```
              Z-axis: Data partitioning
              (shard by user ID, region, etc.)
                         ↑
                         │
Y-axis ─────────────────►│─────────────────────► X-axis
Functional decomposition  │           Horizontal duplication
(split by service:         │           (run N identical copies
 users-service,            │           behind a load balancer)
 orders-service)
```

**X-Axis Scaling** (Horizontal): Clone the service N times. Simple, fast, stateless-friendly.
**Y-Axis Scaling** (Vertical/Functional): Split by domain → microservices. Each service owns its data.
**Z-Axis Scaling** (Data partitioning): Split data by key → shards. Each shard serves a subset.

Most systems start X, then add Y (microservices), then Z (sharding) only when forced.


---
## Vertical Scaling (Scale Up)

**What:** Replace the server with a bigger one (more CPU, RAM, faster SSD).

**When to use:**
- Single-threaded workloads (some databases, legacy apps)
- Simple to implement — no code changes
- Works well up to ~32 cores / ~256GB RAM economically

**❌ PROBLEMS (what breaks without horizontal scaling):**
```
ShopFlow at 1k users: 1 server, 4 cores, 16GB RAM ✓
ShopFlow at 100k users: upgraded to 64 cores, 512GB RAM ✓ (expensive)
ShopFlow at 1M users: server costs $50k/month; maxes out at 96 cores
→ No more vertical headroom. Can't go higher. One machine = single point of failure.
→ Downtime for upgrades: "maintenance windows" every hardware change
```

**Hard limits of vertical scaling:**
- Hardware ceiling (memory bandwidth, cache coherency at many cores)
- Cost curve is exponential beyond mid-tier servers
- Zero redundancy — one machine failure = total outage
- Upgrades require downtime (reboot)


In [ ]:
# Vertical Scaling Simulation: cost curve and diminishing returns

def vertical_scale_cost(rps_needed: int) -> dict:
    '''Simulate what vertical scaling costs vs what you get.'''
    # Approximate: each doubling of server tier costs 2.5x but gives only 2x capacity
    tiers = [
        {"name": "Small",     "rps": 1_000,    "monthly_cost": 100,    "redundancy": False},
        {"name": "Medium",    "rps": 5_000,    "monthly_cost": 500,    "redundancy": False},
        {"name": "Large",     "rps": 20_000,   "monthly_cost": 2_000,  "redundancy": False},
        {"name": "XLarge",    "rps": 50_000,   "monthly_cost": 8_000,  "redundancy": False},
        {"name": "2XLarge",   "rps": 100_000,  "monthly_cost": 25_000, "redundancy": False},
        {"name": "4XLarge",   "rps": 200_000,  "monthly_cost": 80_000, "redundancy": False},
    ]
    for tier in tiers:
        if tier["rps"] >= rps_needed:
            return {**tier, "rps_needed": rps_needed,
                    "cost_per_rps": tier["monthly_cost"] / tier["rps"],
                    "spof": True}
    return {"name": "IMPOSSIBLE", "rps_needed": rps_needed, "spof": True,
            "note": "Vertical scaling maxed out — must go horizontal"}

print("ShopFlow Vertical Scaling Cost Analysis:")
print(f"{'Load':>10} {'Server Tier':>12} {'Monthly Cost':>14} {'Cost/RPS':>10}")
print("-" * 50)
for load in [1_000, 10_000, 50_000, 100_000, 250_000]:
    r = vertical_scale_cost(load)
    if r.get("name") == "IMPOSSIBLE":
        print(f"{load:>10,} {'IMPOSSIBLE':>12}  {'N/A':>13}  {'N/A':>9}")
    else:
        print(f"{load:>10,} {r['name']:>12}  ${r['monthly_cost']:>12,}  ${r['cost_per_rps']:>8.4f}")
print()
print("KEY INSIGHT: cost per RPS INCREASES as you scale up (diminishing returns)")
print("Vertical scaling has a hard ceiling AND zero redundancy (SPOF).")


---
## Horizontal Scaling (Scale Out)

**What:** Add more identical servers. Route traffic across them with a load balancer.
A good horizontal architecture is **stateless** — any server can handle any request.

**✅ WHAT IT BUYS:**
- Linear cost growth (3× servers ≈ 3× capacity)
- No single point of failure (one server dies; others keep serving)
- Zero-downtime deployments (rolling restart)
- Geographic distribution (servers in different regions)

**The Stateless Requirement — The Key Design Constraint:**
```
❌ STATEFUL (breaks horizontal scaling):
   User logs in → server A stores session in local RAM
   Next request routed to server B → "not logged in"!

✅ STATELESS (enables horizontal scaling):
   User logs in → session stored in Redis (shared)
   Any server can serve any request by reading shared Redis
```

**Auto-scaling Pattern (AWS/GCP/Azure):**
```
Monitor: CPU > 70% for 2 minutes → scale out (+1 server)
Monitor: CPU < 30% for 10 minutes → scale in (-1 server)
Min instances: 2 (HA)    Max instances: 50 (cost cap)
Scale-out: fast (60s)    Scale-in: slow (10min, allow drain)
```


In [ ]:
import threading
from collections import defaultdict

class LoadBalancer:
    '''Round-robin load balancer for simulating horizontal scaling.'''
    def __init__(self):
        self._servers: list[str] = []
        self._idx = 0
        self._lock = threading.Lock()
        self._metrics: dict[str, int] = defaultdict(int)

    def add_server(self, name: str): self._servers.append(name)

    def remove_server(self, name: str):
        with self._lock:
            self._servers = [s for s in self._servers if s != name]

    def route(self, request_id: str) -> str | None:
        with self._lock:
            if not self._servers: return None
            server = self._servers[self._idx % len(self._servers)]
            self._idx += 1
            self._metrics[server] += 1
            return server

class AutoScaler:
    '''Simulate auto-scaling based on simulated CPU metrics.'''
    def __init__(self, lb: LoadBalancer, min_servers=2, max_servers=8,
                 scale_out_threshold=0.70, scale_in_threshold=0.30):
        self.lb = lb
        self.min_s, self.max_s = min_servers, max_servers
        self.out_thresh, self.in_thresh = scale_out_threshold, scale_in_threshold
        self._server_count = 0
        for i in range(min_servers):
            self._add()

    def _add(self):
        name = f"web-{self._server_count + 1}"
        self.lb.add_server(name)
        self._server_count += 1
        return name

    def _remove_last(self):
        if self._server_count > self.min_s:
            name = f"web-{self._server_count}"
            self.lb.remove_server(name)
            self._server_count -= 1

    def tick(self, cpu_load: float) -> str:
        action = "steady"
        if cpu_load > self.out_thresh and self._server_count < self.max_s:
            new = self._add()
            action = f"SCALE OUT → added {new} (now {self._server_count} servers)"
        elif cpu_load < self.in_thresh and self._server_count > self.min_s:
            self._remove_last()
            action = f"SCALE IN  → removed server (now {self._server_count} servers)"
        return action

# Simulate ShopFlow's traffic pattern over a day (Black Friday effect)
lb = LoadBalancer()
scaler = AutoScaler(lb)

print("ShopFlow Auto-Scaling Simulation:")
print(f"{'Time':>6} {'CPU%':>6} {'Servers':>8} {'Action'}")
print("-" * 60)
traffic_profile = [(0, 0.3), (6, 0.4), (8, 0.6), (10, 0.8), (12, 0.95),
                   (14, 0.85), (18, 0.75), (22, 0.45), (24, 0.25)]
for hour, cpu in traffic_profile:
    action = scaler.tick(cpu)
    servers = lb._server_count
    print(f"{hour:>4}h {cpu*100:>5.0f}% {servers:>8}  {action}")

print()
# Simulate 100 requests — see even distribution across servers
print("Request distribution (horizontal load balancing):")
for _ in range(100):
    lb.route(f"req_{_}")
for srv, count in sorted(lb._metrics.items()):
    bar = "█" * (count // 2)
    print(f"  {srv}: {count:3d} requests  {bar}")


---
## Scalability Design Patterns

### Database Scaling Path

```
Phase 1: Vertical   → Upgrade DB server (easy, no code changes)
Phase 2: Read       → Add read replicas; route SELECTs to replicas
         Replicas     (works for read-heavy: product catalogue, analytics)
Phase 3: Cache      → Redis/Memcached in front of DB
                      (eliminate repeat reads entirely)
Phase 4: Partition  → Shard writes by user_id, product_id, or region
         (Sharding)   (only when writes saturate a single master)
Phase 5: NoSQL      → Switch hot tables to Cassandra/DynamoDB
                      (when SQL consistency isn't needed for that data)
```

### The Twelve-Factor App Principles for Horizontal Scale

| Factor | Scalability impact |
|---|---|
| Stateless processes | Any server serves any request |
| Config in environment | Same image, different environments |
| Logs as streams | No local log files to manage per server |
| Concurrency via processes | Add more processes, not threads |
| Fast startup/shutdown | Enable rapid scale-out and rolling deploys |

### 🌍 Real-World: How Companies Scaled

| Company | Early architecture | Scaling event | Solution |
|---|---|---|---|
| Twitter (2009) | Monolith, MySQL | Fail Whale (overload on celebrity tweets) | Fan-out service, Redis timeline cache |
| Instagram (2012) | Django monolith, Postgres | Rapid user growth | Read replicas, then sharding by user_id |
| Netflix (2008) | Oracle DB monolith | Outage → decision to re-architect | AWS + microservices + Cassandra |
| Slack (2016) | Shared monolith | Channel message scaling | Vitess (MySQL sharding) + Kafka |

### ⚠️ Gotchas

- **Session affinity breaks horizontal scaling** — store sessions in Redis, not server RAM
- **Database is usually the bottleneck, not the app servers** — scale DB before adding servers
- **Premature sharding** — adds massive complexity; shard only when forced by data volume
- **N+1 query problem** — each horizontal server makes the same DB queries; multiplies load
- **Global state** — file uploads, scheduled jobs, locks must be externalized when you scale
